# Introduction to AI: Customer Segmentation and Prediction
This notebook complements the lecture slides in `slides/introduction_to_ai/slides.md`.

## Story
A wholesaler wants to understand **customer segments** (unsupervised learning) and **predict sales channel** for new customers (supervised learning). We will work with the **Wholesale Customers** dataset (UCI), which contains annual spending across product categories.

**Columns (high level)**
- `channel` (target): 1 = Horeca (Hotels/Restaurants/Cafes), 2 = Retail
- `region`: geographic region (we will ignore it in this exercise)
- spend features: `fresh`, `milk`, `grocery`, `frozen`, `detergents_paper`, `delicassen`

**Goal:** connect clustering and classification to a realistic BI scenario.

## Exercises
- Exercise 1: Explore and prepare the dataset (clean columns, define `X`/`y`).
- Exercise 2: Cluster customers with k-means and interpret segments.
- Exercise 3: Train a classifier to predict `channel` and evaluate it.

Notes:
- Data loading is already provided.
- Work through the notebook from top to bottom.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"

# Load the data (students should not have to do this)
df_raw = pd.read_csv(DATA_URL)

df_raw.head()

In [ ]:
# Exercise 1: Explore and prepare the data

df = df_raw.copy()

df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

display(df.head())
display(df.isna().sum())
print(df.shape)

target_col = "channel"
feature_cols = [c for c in df.columns if c not in [target_col, "region"]]

X = df[feature_cols]
y = df[target_col]

display(X.describe())
y.value_counts()

## Part A: Unsupervised learning - k-means clustering
We ignore the labels and group customers purely by spending patterns. This mimics a real **segmentation** use case.

### Tasks
1. Scale the features.
2. Fit k-means with a chosen k (start with k=3).
3. Inspect cluster sizes and compare clusters with the true `channel`.

In [ ]:
# Exercise 2: K-means clustering

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 3
kmeans = KMeans(n_clusters=k, n_init=20, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

df_clustered = df.copy()
df_clustered["cluster"] = clusters

display(df_clustered["cluster"].value_counts().sort_index())
display(pd.crosstab(df_clustered["cluster"], df_clustered[target_col]))

In [ ]:
# Visualize clusters in 2D with PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df_plot = df_clustered.copy()
df_plot["pc1"] = X_pca[:, 0]
df_plot["pc2"] = X_pca[:, 1]

plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    df_plot["pc1"],
    df_plot["pc2"],
    c=df_plot["cluster"],
    cmap="viridis",
    alpha=0.8,
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-means clusters (PCA projection)")
plt.colorbar(scatter, label="cluster")
plt.show()

## Part B: Supervised learning - classification
Now we use the known `channel` labels to **predict** whether a new customer is Horeca or Retail.

### Tasks
1. Split the data into train and test sets.
2. Build a pipeline with scaling + a classifier.
3. Train, predict, and evaluate.

In [ ]:
# Exercise 3: Classification

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=300)),
])

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# Plot the confusion matrix for your classifier

ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test, cmap="Blues")
plt.title("Confusion matrix")
plt.show()

## Discussion and extensions
- Try different values of k and justify your choice.
- Compare clusters to the true `channel` and discuss mismatches.
- Swap the classifier (DecisionTree, RandomForest, SVM) and compare metrics.
- Compare performance with and without feature scaling.
- If you had real business data, which additional features would help?